# DistilBERT on SST-2 Experiment Console

This notebook is the unified entry point for the repository. It now supports running a four-seed experiment suite end-to-end in one pass while keeping result inspection inside the same notebook. The training, evaluation, and token-importance analysis logic stays in importable Python modules under `src/model`.


## Environment And Dependency Check

Run this cell first to locate the project root, expose the package root on `sys.path`, and inspect the core package versions.


In [1]:
import json
import sys
from pathlib import Path

import datasets
import pandas as pd
import plotly.express as px
import torch
import transformers
from IPython.display import HTML, Markdown, display


def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "src" / "model").exists() and (candidate / "experiment.ipynb").exists():
            return candidate
    raise RuntimeError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.model import AnalysisConfig, EvalConfig, TrainConfig, evaluate, run_attention_analysis, train

pd.set_option("display.max_colwidth", 160)

versions_df = pd.DataFrame(
    [
        {"package": "python", "version": sys.version.split()[0]},
        {"package": "torch", "version": torch.__version__},
        {"package": "transformers", "version": transformers.__version__},
        {"package": "datasets", "version": datasets.__version__},
        {"package": "pandas", "version": pd.__version__},
    ]
)
display(versions_df)
print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")


d:\5329_assignment2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,package,version
0,python,3.13.12
1,torch,2.10.0+cu128
2,transformers,4.57.6
3,datasets,2.21.0
4,pandas,2.3.3


Project root: D:\5329_assignment2
Python executable: d:\5329_assignment2\.venv\Scripts\python.exe


## Configuration

Edit the shared settings, the `SEEDS` list, and `INSPECT_SEED` below. The notebook builds one `TrainConfig`, `EvalConfig`, and `AnalysisConfig` per seed, writes artifacts to seed-specific directories, and later aggregates the saved results across seeds.

- `TrainConfig` contains training-only settings.
- `EvalConfig` contains evaluation-only settings.
- `AnalysisConfig` contains token-importance-analysis-only settings.

To reuse existing checkpoints for all seeds, skip the training cell and run only the evaluation, analysis, or loading sections.


In [2]:
EXPERIMENT_NAME = "distilbert_sst2_multiseed"
SEEDS = [42, 43, 44, 45]
INSPECT_SEED = 42

COMMON_SETTINGS = {
    "model_name": "distilbert-base-uncased",
    "cache_dir": PROJECT_ROOT / ".cache" / "huggingface",
    "max_length": 128,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "num_train_epochs": 2.0,
    "per_device_train_batch_size": 16,
    "per_device_eval_batch_size": 32,
    "logging_steps": 50,
    "save_total_limit": 2,
    "metric_for_best_model": "accuracy",
    "max_train_examples": None,
    "max_eval_examples": None,
    "split": "validation",
    "sample_size": 1000000,
    "analysis_subset": "all",
    "top_k_values": [1, 2, 3],
    "top_ratios": [0.05, 0.1, 0.2],
    "random_trials": 5,
    "max_validation_examples": None,
    "case_study_count": 5,
}


def build_seed_dirs(experiment_name: str, seed: int) -> dict[str, Path]:
    run_name = f"{experiment_name}_seed{seed}"
    return {
        "run_name": run_name,
        "checkpoint_dir": PROJECT_ROOT / "results" / "checkpoints" / run_name,
        "metrics_dir": PROJECT_ROOT / "results" / "metrics" / run_name,
        "analysis_dir": PROJECT_ROOT / "results" / "analysis" / run_name,
    }



def build_experiment_bundle(seed: int) -> dict:
    seed_dirs = build_seed_dirs(EXPERIMENT_NAME, seed)
    train_config = TrainConfig(
        model_name=COMMON_SETTINGS["model_name"],
        cache_dir=str(COMMON_SETTINGS["cache_dir"]),
        output_dir=str(seed_dirs["checkpoint_dir"]),
        metrics_dir=str(seed_dirs["metrics_dir"]),
        max_length=COMMON_SETTINGS["max_length"],
        seed=seed,
        learning_rate=COMMON_SETTINGS["learning_rate"],
        weight_decay=COMMON_SETTINGS["weight_decay"],
        num_train_epochs=COMMON_SETTINGS["num_train_epochs"],
        per_device_train_batch_size=COMMON_SETTINGS["per_device_train_batch_size"],
        per_device_eval_batch_size=COMMON_SETTINGS["per_device_eval_batch_size"],
        logging_steps=COMMON_SETTINGS["logging_steps"],
        save_total_limit=COMMON_SETTINGS["save_total_limit"],
        metric_for_best_model=COMMON_SETTINGS["metric_for_best_model"],
        max_train_examples=COMMON_SETTINGS["max_train_examples"],
        max_eval_examples=COMMON_SETTINGS["max_eval_examples"],
    )

    eval_config = EvalConfig(
        checkpoint_dir=train_config.output_dir,
        cache_dir=train_config.cache_dir,
        metrics_dir=train_config.metrics_dir,
        split=COMMON_SETTINGS["split"],
        max_length=train_config.max_length,
        seed=train_config.seed,
        per_device_eval_batch_size=train_config.per_device_eval_batch_size,
        max_eval_examples=COMMON_SETTINGS["max_eval_examples"],
    )

    analysis_config = AnalysisConfig(
        checkpoint_dir=train_config.output_dir,
        cache_dir=train_config.cache_dir,
        metrics_dir=train_config.metrics_dir,
        analysis_dir=str(seed_dirs["analysis_dir"]),
        max_length=train_config.max_length,
        seed=train_config.seed,
        sample_size=COMMON_SETTINGS["sample_size"],
        analysis_subset=COMMON_SETTINGS["analysis_subset"],
        top_k_values=COMMON_SETTINGS["top_k_values"],
        top_ratios=COMMON_SETTINGS["top_ratios"],
        random_trials=COMMON_SETTINGS["random_trials"],
        max_validation_examples=COMMON_SETTINGS["max_validation_examples"],
        case_study_count=COMMON_SETTINGS["case_study_count"],
    )

    artifact_paths = {
        "train_metrics": Path(train_config.metrics_dir) / "train_metrics.json",
        "run_config": Path(train_config.metrics_dir) / "run_config.json",
        "validation_metrics": Path(eval_config.metrics_dir) / f"{eval_config.split}_metrics.json",
        "validation_predictions": Path(eval_config.metrics_dir) / f"{eval_config.split}_predictions.csv",
        "ranking_similarity": Path(analysis_config.analysis_dir) / "ranking_similarity_summary.csv",
        "deletion_summary": Path(analysis_config.analysis_dir) / "deletion_summary.csv",
        "deletion_records": Path(analysis_config.analysis_dir) / "deletion_records.csv",
        "qualitative_cases": Path(analysis_config.analysis_dir) / "qualitative_cases.md",
    }

    return {
        "seed": seed,
        "run_name": seed_dirs["run_name"],
        "dirs": seed_dirs,
        "train_config": train_config,
        "eval_config": eval_config,
        "analysis_config": analysis_config,
        "artifact_paths": artifact_paths,
    }


experiments = [build_experiment_bundle(seed) for seed in SEEDS]
experiment_index = {experiment["seed"]: experiment for experiment in experiments}
if INSPECT_SEED not in experiment_index:
    raise ValueError(f"INSPECT_SEED={INSPECT_SEED} must be one of {SEEDS}.")

inspect_experiment = experiment_index[INSPECT_SEED]
train_config = inspect_experiment["train_config"]
eval_config = inspect_experiment["eval_config"]
analysis_config = inspect_experiment["analysis_config"]

shared_settings_df = pd.DataFrame(
    [
        {"scope": "shared", "setting": "experiment_name", "value": EXPERIMENT_NAME},
        {"scope": "shared", "setting": "seeds", "value": str(SEEDS)},
        {"scope": "shared", "setting": "inspect_seed", "value": INSPECT_SEED},
        {"scope": "train", "setting": "model_name", "value": COMMON_SETTINGS["model_name"]},
        {"scope": "train", "setting": "epochs", "value": COMMON_SETTINGS["num_train_epochs"]},
        {"scope": "train", "setting": "train_batch_size", "value": COMMON_SETTINGS["per_device_train_batch_size"]},
        {"scope": "train", "setting": "eval_batch_size", "value": COMMON_SETTINGS["per_device_eval_batch_size"]},
        {"scope": "train", "setting": "learning_rate", "value": COMMON_SETTINGS["learning_rate"]},
        {"scope": "analysis", "setting": "sample_size", "value": COMMON_SETTINGS["sample_size"]},
        {"scope": "analysis", "setting": "analysis_subset", "value": COMMON_SETTINGS["analysis_subset"]},
        {"scope": "analysis", "setting": "top_k_values", "value": str(COMMON_SETTINGS["top_k_values"])},
        {"scope": "analysis", "setting": "top_ratios", "value": str(COMMON_SETTINGS["top_ratios"])},
        {"scope": "analysis", "setting": "random_trials", "value": COMMON_SETTINGS["random_trials"]},
    ]
)

action_rows = []
for experiment in experiments:
    action_rows.append(
        {
            "seed": experiment["seed"],
            "run_name": experiment["run_name"],
            "checkpoint_dir": str(experiment["dirs"]["checkpoint_dir"]),
            "metrics_dir": str(experiment["dirs"]["metrics_dir"]),
            "analysis_dir": str(experiment["dirs"]["analysis_dir"]),
        }
    )

display(shared_settings_df)
display(pd.DataFrame(action_rows).sort_values("seed").reset_index(drop=True))


,scope,setting,value
0,shared,experiment_name,distilbert_sst2_multiseed
1,shared,seeds,"[42, 43, 44, 45]"
2,shared,inspect_seed,42
3,train,model_name,distilbert-base-uncased
4,train,epochs,2.0
5,train,train_batch_size,16
6,train,eval_batch_size,32
7,train,learning_rate,0.00002
8,analysis,sample_size,1000000
9,analysis,analysis_subset,all


,seed,run_name,checkpoint_dir,metrics_dir,analysis_dir
0,42,distilbert_sst2_multiseed_seed42,D:\5329_assignment2\results\checkpoints\distilbert_sst2_multiseed_seed42,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed42,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed42
1,43,distilbert_sst2_multiseed_seed43,D:\5329_assignment2\results\checkpoints\distilbert_sst2_multiseed_seed43,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed43,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed43
2,44,distilbert_sst2_multiseed_seed44,D:\5329_assignment2\results\checkpoints\distilbert_sst2_multiseed_seed44,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed44,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed44
3,45,distilbert_sst2_multiseed_seed45,D:\5329_assignment2\results\checkpoints\distilbert_sst2_multiseed_seed45,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed45,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed45


## Training Stage

Run this cell to fine-tune all configured seeds sequentially.


In [3]:
train_results = {}
train_summary_rows = []

for experiment in experiments:
    seed = experiment["seed"]
    print(f"=== Seed {seed}: training ===")
    train_result = train(experiment["train_config"])
    train_results[seed] = train_result
    train_summary_rows.append(
        {
            "seed": seed,
            "best_metric": train_result.metrics.get("best_metric"),
            "train_loss": train_result.metrics.get("train_metrics", {}).get("train_loss"),
            "train_runtime": train_result.metrics.get("train_metrics", {}).get("train_runtime"),
            "metrics_path": str(train_result.metrics_path),
            "best_checkpoint_dir": str(train_result.best_checkpoint_dir or train_result.checkpoint_dir),
        }
    )

train_summary_df = pd.DataFrame(train_summary_rows).sort_values("seed").reset_index(drop=True)
display(train_summary_df)


=== Seed 42: training ===
[Train] Starting fine-tuning with model=distilbert-base-uncased, epochs=2.0, train_batch_size=16, eval_batch_size=32, seed=42
[Train] Loading SST-2 dataset from cache_dir=D:\5329_assignment2\.cache\huggingface
[Train] Loading tokenizer and tokenizing dataset


Tokenizing SST-2: 100%|██████████| 1821/1821 [00:00<00:00, 36968.80 examples/s]

[Train] Prepared datasets: train_examples=67349, validation_examples=872
[Train] Loading pretrained model



Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
D:\5329_assignment2\src\model\api.py:262: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


[Train] Launching Trainer.train(); progress bar will update during optimization


...

D:\5329_assignment2\src\model\api.py:262: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.197900,0.309961,0.894495,0.898901
2,0.101500,0.379009,0.900229,0.902793


[Train] Running final validation on the selected eval split


[Train] Saving checkpoint to D:\5329_assignment2\results\checkpoints\distilbert_sst2_multiseed_seed45
[Train] Completed training. best_metric=0.9002293577981652, best_checkpoint=D:\5329_assignment2\results\checkpoints\distilbert_sst2_multiseed_seed45\checkpoint-8420, metrics_path=D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed45\train_metrics.json


,seed,best_metric,train_loss,train_runtime,metrics_path,best_checkpoint_dir
0,42,0.912844,0.176689,268.8121,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed42\train_metrics.json,D:\5329_assignment2\results\checkpoints\distilbert_sst2_multiseed_seed42\checkpoint-8420
1,43,0.904817,0.175950,266.9646,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed43\train_metrics.json,D:\5329_assignment2\results\checkpoints\distilbert_sst2_multiseed_seed43\checkpoint-8420
2,44,0.911697,0.175440,267.3477,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed44\train_metrics.json,D:\5329_assignment2\results\checkpoints\distilbert_sst2_multiseed_seed44\checkpoint-4210
3,45,0.900229,0.177532,268.4285,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed45\train_metrics.json,D:\5329_assignment2\results\checkpoints\distilbert_sst2_multiseed_seed45\checkpoint-8420


## Evaluation Stage

Run this cell to evaluate the saved checkpoint for every configured seed.


In [4]:
eval_results = {}
eval_summary_rows = []

for experiment in experiments:
    seed = experiment["seed"]
    print(f"=== Seed {seed}: evaluation ===")
    eval_result = evaluate(experiment["eval_config"])
    eval_results[seed] = eval_result
    eval_summary_rows.append({"seed": seed, **eval_result.metrics, "metrics_path": str(eval_result.metrics_path)})

eval_summary_df = pd.DataFrame(eval_summary_rows).sort_values("seed").reset_index(drop=True)
display(eval_summary_df)


=== Seed 42: evaluation ===
[Eval] Starting evaluation for checkpoint_dir=D:\5329_assignment2\results\checkpoints\distilbert_sst2_multiseed_seed42 on split=validation
[Eval] Loading SST-2 split with cache_dir=D:\5329_assignment2\.cache\huggingface
[Eval] Loading tokenizer and checkpoint
[Eval] Using device=cuda; num_examples=872
[Eval] Tokenizing evaluation split


Evaluating validation: 100%|██████████| 28/28 [00:00<00:00, 84.70batch/s, avg_loss=0.3180]

[Eval] Completed evaluation. accuracy=0.9128, f1=0.9152, loss=0.3180, metrics_path=D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed42\validation_metrics.json
=== Seed 43: evaluation ===
[Eval] Starting evaluation for checkpoint_dir=D:\5329_assignment2\results\checkpoints\distilbert_sst2_multiseed_seed43 on split=validation
[Eval] Loading SST-2 split with cache_dir=D:\5329_assignment2\.cache\huggingface
[Eval] Loading tokenizer and checkpoint


[Eval] Using device=cuda; num_examples=872
[Eval] Tokenizing evaluation split


Evaluating validation: 100%|██████████| 28/28 [00:00<00:00, 84.51batch/s, avg_loss=0.3474]

...

[Eval] Completed evaluation. accuracy=0.9117, f1=0.9108, loss=0.2759, metrics_path=D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed44\validation_metrics.json
=== Seed 45: evaluation ===
[Eval] Starting evaluation for checkpoint_dir=D:\5329_assignment2\results\checkpoints\distilbert_sst2_multiseed_seed45 on split=validation
[Eval] Loading SST-2 split with cache_dir=D:\5329_assignment2\.cache\huggingface
[Eval] Loading tokenizer and checkpoint


[Eval] Using device=cuda; num_examples=872
[Eval] Tokenizing evaluation split


Evaluating validation: 100%|██████████| 28/28 [00:00<00:00, 84.43batch/s, avg_loss=0.3766]

[Eval] Completed evaluation. accuracy=0.9002, f1=0.9028, loss=0.3766, metrics_path=D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed45\validation_metrics.json


,seed,accuracy,f1,loss,num_examples,metrics_path
0,42,0.912844,0.915179,0.318047,872,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed42\validation_metrics.json
1,43,0.904817,0.908891,0.347401,872,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed43\validation_metrics.json
2,44,0.911697,0.910776,0.275913,872,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed44\validation_metrics.json
3,45,0.900229,0.902793,0.376587,872,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed45\validation_metrics.json


## Importance Analysis Stage

Run this cell to produce token-importance rankings, ranking-similarity tables, deletion records, and qualitative cases for every configured seed.


In [3]:
analysis_results = {}
analysis_summary_rows = []

for experiment in experiments:
    seed = experiment["seed"]
    print(f"=== Seed {seed}: importance analysis ===")
    analysis_result = run_attention_analysis(experiment["analysis_config"])
    analysis_results[seed] = analysis_result
    analysis_summary_rows.append(
        {
            "seed": seed,
            "analysis_dir": str(analysis_result.analysis_dir),
            "ranking_similarity_summary": str(analysis_result.ranking_similarity_summary_path),
            "deletion_summary": str(analysis_result.deletion_summary_path),
            "qualitative_cases": str(analysis_result.qualitative_cases_path),
        }
    )

analysis_summary_df = pd.DataFrame(analysis_summary_rows).sort_values("seed").reset_index(drop=True)
display(analysis_summary_df)


=== Seed 42: importance analysis ===
[Analysis] Starting token-importance analysis with sample_size=1000000, analysis_subset=all, top_k_values=[1, 2, 3], random_trials=5
[Analysis] Loading tokenizer and checkpoint for analysis
[Analysis] Using device=cuda
[Analysis] Loading validation split from cache_dir=D:\5329_assignment2\.cache\huggingface
[Analysis] Scanning 872 validation examples to build the candidate pool


Filtering analysis candidates: 100%|██████████| 872/872 [00:04<00:00, 192.64example/s, candidates=872]

[Analysis] Candidate pool ready: 872 examples; sampled 872 for ranking and deletion analysis



Analyzing sampled examples: 100%|██████████| 872/872 [03:19<00:00,  4.36sample/s, records=20920]


[Analysis] Completed analysis. similarity_summary=D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed42\ranking_similarity_summary.csv, deletion_summary=D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed42\deletion_summary.csv, qualitative_cases=D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed42\qualitative_cases.md
=== Seed 43: importance analysis ===
[Analysis] Starting token-importance analysis with sample_size=1000000, analysis_subset=all, top_k_values=[1, 2, 3], random_trials=5
[Analysis] Loading tokenizer and checkpoint for analysis
[Analysis] Using device=cuda
[Analysis] Loading validation split from cache_dir=D:\5329_assignment2\.cache\huggingface
[Analysis] Scanning 872 validation examples to build the candidate pool


...

Filtering analysis candidates: 100%|██████████| 872/872 [00:03<00:00, 223.44example/s, candidates=872]

[Analysis] Candidate pool ready: 872 examples; sampled 872 for ranking and deletion analysis



Analyzing sampled examples: 100%|██████████| 872/872 [03:15<00:00,  4.45sample/s, records=20920]


[Analysis] Completed analysis. similarity_summary=D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed45\ranking_similarity_summary.csv, deletion_summary=D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed45\deletion_summary.csv, qualitative_cases=D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed45\qualitative_cases.md


,seed,analysis_dir,ranking_similarity_summary,deletion_summary,qualitative_cases
0,42,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed42,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed42\ranking_similarity_summary.csv,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed42\deletion_summary.csv,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed42\qualitative_cases.md
1,43,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed43,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed43\ranking_similarity_summary.csv,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed43\deletion_summary.csv,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed43\qualitative_cases.md
2,44,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed44,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed44\ranking_similarity_summary.csv,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed44\deletion_summary.csv,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed44\qualitative_cases.md
3,45,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed45,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed45\ranking_similarity_summary.csv,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed45\deletion_summary.csv,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed45\qualitative_cases.md


## Result Loading Helpers

The next helpers load seed-specific artifacts, build cross-seed summaries, and keep the notebook usable when you rerun only selected stages.


In [4]:
def load_json_if_exists(path: Path):
    if not path.exists():
        print(f"Missing file: {path}")
        return None
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)



def load_csv_if_exists(path: Path):
    if not path.exists():
        print(f"Missing file: {path}")
        return None
    return pd.read_csv(path)



def display_markdown_file(path: Path):
    if not path.exists():
        print(f"Missing file: {path}")
        return
    display(Markdown(path.read_text(encoding="utf-8")))



def display_plotly_figure(fig):
    try:
        fig.show()
    except ValueError as exc:
        if "nbformat" not in str(exc):
            raise
        print("Plotly renderer fallback: nbformat is missing, displaying inline HTML instead.")
        display(HTML(fig.to_html(include_plotlyjs="cdn")))



def load_seed_json_rows(artifact_name: str):
    rows = []
    for seed in SEEDS:
        payload = load_json_if_exists(artifact_paths_by_seed[seed][artifact_name])
        if payload is None:
            continue
        rows.append({"seed": seed, **payload})
    if not rows:
        return None
    return pd.DataFrame(rows)



def concat_seed_csv(artifact_name: str):
    frames = []
    for seed in SEEDS:
        frame = load_csv_if_exists(artifact_paths_by_seed[seed][artifact_name])
        if frame is None:
            continue
        frame = frame.copy()
        frame.insert(0, "seed", seed)
        frames.append(frame)
    if not frames:
        return None
    return pd.concat(frames, ignore_index=True)


artifact_paths_by_seed = {
    experiment["seed"]: experiment["artifact_paths"]
    for experiment in experiments
}
inspect_artifact_paths = artifact_paths_by_seed[INSPECT_SEED]

artifact_inventory_rows = []
for seed in SEEDS:
    for artifact_name, path in artifact_paths_by_seed[seed].items():
        artifact_inventory_rows.append(
            {
                "seed": seed,
                "artifact": artifact_name,
                "path": str(path),
                "exists": path.exists(),
            }
        )

artifact_inventory_df = pd.DataFrame(artifact_inventory_rows).sort_values(["seed", "artifact"]).reset_index(drop=True)
display(artifact_inventory_df)
print(f"Inspecting detailed outputs for seed {INSPECT_SEED}.")


,seed,artifact,path,exists
0,42,deletion_records,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed42\deletion_records.csv,True
1,42,deletion_summary,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed42\deletion_summary.csv,True
2,42,qualitative_cases,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed42\qualitative_cases.md,True
3,42,ranking_similarity,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed42\ranking_similarity_summary.csv,True
4,42,run_config,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed42\run_config.json,True
5,42,train_metrics,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed42\train_metrics.json,True
6,42,validation_metrics,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed42\validation_metrics.json,True
7,42,validation_predictions,D:\5329_assignment2\results\metrics\distilbert_sst2_multiseed_seed42\validation_predictions.csv,True
8,43,deletion_records,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed43\deletion_records.csv,True
9,43,deletion_summary,D:\5329_assignment2\results\analysis\distilbert_sst2_multiseed_seed43\deletion_summary.csv,True


Inspecting detailed outputs for seed 42.


## Validation Metrics, Aggregate Summary, And Prediction Sample


In [7]:
validation_metrics_df = load_seed_json_rows("validation_metrics")
if validation_metrics_df is not None:
    validation_metrics_df = validation_metrics_df.sort_values("seed").reset_index(drop=True)
    display(validation_metrics_df)

    aggregate_rows = []
    for metric_name in ["accuracy", "f1", "loss", "num_examples"]:
        if metric_name not in validation_metrics_df.columns:
            continue
        aggregate_rows.append(
            {
                "metric": metric_name,
                "mean": validation_metrics_df[metric_name].mean(),
                "std": validation_metrics_df[metric_name].std(ddof=0),
                "min": validation_metrics_df[metric_name].min(),
                "max": validation_metrics_df[metric_name].max(),
            }
        )
    if aggregate_rows:
        display(pd.DataFrame(aggregate_rows))

prediction_sample = load_csv_if_exists(inspect_artifact_paths["validation_predictions"])
if prediction_sample is not None:
    print(f"Prediction sample for seed {INSPECT_SEED}")
    display(prediction_sample.head(10))


,seed,accuracy,f1,loss,num_examples
0,42,0.912844,0.915179,0.318047,872
1,43,0.904817,0.908891,0.347401,872
2,44,0.911697,0.910776,0.275913,872
3,45,0.900229,0.902793,0.376587,872


,metric,mean,std,min,max
0,accuracy,0.907397,0.005153,0.900229,0.912844
1,f1,0.909410,0.004450,0.902793,0.915179
2,loss,0.329487,0.037217,0.275913,0.376587
3,num_examples,872.000000,0.000000,872.000000,872.000000


Prediction sample for seed 42


,validation_index,sentence,label,label_name,prediction,prediction_name,correct,prob_negative,prob_positive,confidence
0,0,it 's a charming and often affecting journey .,1,positive,1,positive,True,0.000403,0.999597,0.999597
1,1,unflinchingly bleak and desperate,0,negative,0,negative,True,0.996670,0.003330,0.996670
2,2,allows us to hope that nolan is poised to embark a major career as a commercial yet inventive filmmaker .,1,positive,1,positive,True,0.000905,0.999095,0.999095
3,3,"the acting , costumes , music , cinematography and sound are all astounding given the production 's austere locales .",1,positive,1,positive,True,0.002463,0.997537,0.997537
4,4,"it 's slow -- very , very slow .",0,negative,0,negative,True,0.997041,0.002959,0.997041
5,5,"although laced with humor and a few fanciful touches , the film is a refreshingly serious look at young women .",1,positive,1,positive,True,0.000613,0.999387,0.999387
6,6,a sometimes tedious film .,0,negative,0,negative,True,0.995579,0.004421,0.995579
7,7,or doing last year 's taxes with your ex-wife .,0,negative,0,negative,True,0.988557,0.011443,0.988557
8,8,you do n't have to know about music to appreciate the film 's easygoing blend of comedy and romance .,1,positive,1,positive,True,0.008843,0.991157,0.991157
9,9,"in exactly 89 minutes , most of which passed as slowly as if i 'd been sitting naked on an igloo , formula 51 sank from quirky to jerky to utter turkey .",0,negative,0,negative,True,0.996142,0.003858,0.996142


## Ranking Similarity Across Seeds


In [5]:
ranking_similarity_df = concat_seed_csv("ranking_similarity")
if ranking_similarity_df is not None:
    ranking_similarity_df = ranking_similarity_df.sort_values(["method_a", "method_b", "seed"]).reset_index(drop=True)
    display(ranking_similarity_df)

    ranking_similarity_aggregate_df = (
        ranking_similarity_df.groupby(["method_a", "method_b"], dropna=False)
        .agg(
            num_seeds=("seed", "nunique"),
            mean_ranked_token_count_mean=("mean_ranked_token_count", "mean"),
            mean_ranked_token_count_std=("mean_ranked_token_count", "std"),
            mean_spearman_rank_correlation_mean=("mean_spearman_rank_correlation", "mean"),
            mean_spearman_rank_correlation_std=("mean_spearman_rank_correlation", "std"),
        )
        .reset_index()
    )
    ranking_similarity_aggregate_df["mean_ranked_token_count_std"] = ranking_similarity_aggregate_df[
        "mean_ranked_token_count_std"
    ].fillna(0.0)
    ranking_similarity_aggregate_df["mean_spearman_rank_correlation_std"] = ranking_similarity_aggregate_df[
        "mean_spearman_rank_correlation_std"
    ].fillna(0.0)
    display(ranking_similarity_aggregate_df)


,seed,method_a,method_b,num_examples,mean_ranked_token_count,mean_spearman_rank_correlation,std_spearman_rank_correlation
0,42,attention,grad_x_input,872,23.163991,0.240324,0.257557
1,43,attention,grad_x_input,872,23.163991,0.225775,0.262704
2,44,attention,grad_x_input,872,23.163991,0.211799,0.265403
3,45,attention,grad_x_input,872,23.163991,0.217725,0.265717
4,42,attention,loo,872,23.163991,0.174863,0.263084
5,43,attention,loo,872,23.163991,0.175089,0.256193
6,44,attention,loo,872,23.163991,0.177022,0.268027
7,45,attention,loo,872,23.163991,0.177558,0.263910
8,42,grad_x_input,loo,872,23.163991,0.082794,0.261580
9,43,grad_x_input,loo,872,23.163991,0.097671,0.263777


,method_a,method_b,num_seeds,mean_ranked_token_count_mean,mean_ranked_token_count_std,mean_spearman_rank_correlation_mean,mean_spearman_rank_correlation_std
0,attention,grad_x_input,4,23.163991,0.0,0.223906,0.012353
1,attention,loo,4,23.163991,0.0,0.176133,0.001357
2,grad_x_input,loo,4,23.163991,0.0,0.087480,0.007430


## Deletion Summary Across Seeds


In [6]:
deletion_summary_df = concat_seed_csv("deletion_summary")
if deletion_summary_df is not None:
    deletion_summary_df = deletion_summary_df.sort_values(
        ["method", "k_setting_type", "k_setting_value", "seed"]
    ).reset_index(drop=True)
    display(deletion_summary_df)

    deletion_aggregate_df = (
        deletion_summary_df.groupby(["method", "k_setting_type", "k_setting_value", "actual_k"], dropna=False)
        .agg(
            num_seeds=("seed", "nunique"),
            mean_confidence_drop_mean=("mean_confidence_drop", "mean"),
            mean_confidence_drop_std=("mean_confidence_drop", "std"),
            mean_gold_label_prob_drop_mean=("mean_gold_label_prob_drop", "mean"),
            mean_gold_label_prob_drop_std=("mean_gold_label_prob_drop", "std"),
            mean_target_prob_drop_mean=("mean_target_prob_drop", "mean"),
            mean_target_prob_drop_std=("mean_target_prob_drop", "std"),
            flip_rate_mean=("flip_rate", "mean"),
            flip_rate_std=("flip_rate", "std"),
            becomes_incorrect_rate_mean=("becomes_incorrect_rate", "mean"),
            becomes_incorrect_rate_std=("becomes_incorrect_rate", "std"),
        )
        .reset_index()
    )
    for column in deletion_aggregate_df.columns:
        if column.endswith("_std"):
            deletion_aggregate_df[column] = deletion_aggregate_df[column].fillna(0.0)
    display(deletion_aggregate_df)

    fig = px.line(
        deletion_aggregate_df,
        x="actual_k",
        y="mean_confidence_drop_mean",
        color="method",
        line_dash="k_setting_type",
        markers=True,
        error_y="mean_confidence_drop_std",
        title="Confidence Drop After Token Deletion (mean +/- std across seeds)",
        labels={
            "actual_k": "Deleted token count",
            "mean_confidence_drop_mean": "Mean confidence drop across seeds",
            "method": "Deletion method",
            "k_setting_type": "Deletion budget type",
        },
    )
    display_plotly_figure(fig)


,seed,method,k_setting_type,k_setting_value,actual_k,num_examples,mean_confidence_drop,std_confidence_drop,mean_gold_label_prob_drop,mean_target_prob_drop,flip_rate,becomes_incorrect_rate,mean_delta_prob_negative,mean_delta_prob_positive
0,42,attention,ratio,0.05,1,381,0.010833,0.090518,0.117272,0.156558,0.170604,0.194226,-0.099571,0.099571
1,42,attention,ratio,0.05,2,441,0.016445,0.092084,0.091848,0.121931,0.131519,0.190476,-0.047354,0.047354
2,42,attention,ratio,0.05,3,50,-0.010140,0.153164,0.128709,0.086870,0.120000,0.260000,-0.118813,0.118813
3,43,attention,ratio,0.05,1,381,0.019505,0.090147,0.107683,0.163816,0.170604,0.188976,-0.087086,0.087086
4,43,attention,ratio,0.05,2,441,0.016796,0.098459,0.101608,0.157029,0.174603,0.208617,-0.057688,0.057688
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
363,45,random,top_k,2.00,2,872,0.005141,0.041633,0.028368,0.052999,0.063991,0.126606,-0.010336,0.010336
364,42,random,top_k,3.00,3,870,0.007032,0.057109,0.051741,0.076191,0.093333,0.143908,-0.023330,0.023330
365,43,random,top_k,3.00,3,870,0.009874,0.053598,0.042471,0.076170,0.085287,0.135632,-0.008505,0.008505
366,44,random,top_k,3.00,3,870,0.013340,0.065642,0.054281,0.070436,0.087356,0.147356,0.004317,-0.004317


,method,k_setting_type,k_setting_value,actual_k,num_seeds,mean_confidence_drop_mean,mean_confidence_drop_std,mean_gold_label_prob_drop_mean,mean_gold_label_prob_drop_std,mean_target_prob_drop_mean,mean_target_prob_drop_std,flip_rate_mean,flip_rate_std,becomes_incorrect_rate_mean,becomes_incorrect_rate_std
0,attention,ratio,0.05,1,4,0.015616,0.005451,0.118297,0.012049,0.159078,0.004945,0.171260,0.011811,0.203412,0.015229
1,attention,ratio,0.05,2,4,0.020773,0.006748,0.104255,0.017599,0.146767,0.016941,0.157596,0.019462,0.201814,0.016766
2,attention,ratio,0.05,3,4,0.004836,0.018661,0.114245,0.016268,0.126035,0.045553,0.150000,0.047610,0.250000,0.011547
3,attention,ratio,0.10,1,4,0.020899,0.010118,0.181383,0.030216,0.209874,0.019546,0.220874,0.021528,0.266990,0.028027
4,attention,ratio,0.10,2,4,0.018323,0.008326,0.155194,0.015591,0.217772,0.015681,0.237410,0.017126,0.239209,0.016220
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,random,ratio,0.20,10,4,-0.012657,0.021350,-0.006799,0.025472,0.070461,0.033564,0.092857,0.029738,0.107143,0.018443
88,random,ratio,0.20,11,4,0.004193,0.017524,0.088403,0.029032,0.088403,0.029032,0.107143,0.027355,0.107143,0.027355
89,random,top_k,1.00,1,4,0.002952,0.002048,0.014419,0.001791,0.028504,0.004973,0.038303,0.002060,0.107970,0.004341
90,random,top_k,2.00,2,4,0.005649,0.002419,0.032219,0.002989,0.052696,0.003068,0.064564,0.000577,0.125401,0.002304


## Qualitative Cases For `INSPECT_SEED`


In [7]:
print(f"Qualitative cases for seed {INSPECT_SEED}")
display_markdown_file(inspect_artifact_paths["qualitative_cases"])


Qualitative cases for seed 42


# Qualitative Importance Cases

Attention importance is defined as the last-layer mean attention from `[CLS]` to each non-special token.
Gradient × input is computed against the original predicted-class logit and ranked by absolute attribution magnitude.
Leave-one-out importance is defined as the drop in original predicted-class probability after deleting one token at a time.
Tokens remain in wordpiece form for this DistilBERT SST-2 implementation.

## Validation Example 519

- Sentence: moretti 's compelling anatomy of grief and the difficult process of adapting to loss . 
- Gold label: negative
- Base prediction: positive
- Base probabilities: negative=0.0017, positive=0.9983
- Case deletion setting: top_k=1.0 (actual_k=1)

### attention

- Top tokens: . (0.2792), compelling (0.1938), s (0.1305), ' (0.0973), and (0.0462), ##tti (0.0228), anatomy (0.0213), more (0.0190)
- Deleted tokens: .
- New probabilities after deletion: negative=0.0016, positive=0.9984
- Confidence drop: -0.0000
- Target probability drop: -0.0000
- Prediction flipped: False

### grad_x_input

- Top tokens: anatomy (0.1097), ' (0.0628), compelling (0.0558), s (0.0459), more (0.0380), grief (0.0367), the (0.0231), adapting (0.0218)
- Deleted tokens: anatomy
- New probabilities after deletion: negative=0.0031, positive=0.9969
- Confidence drop: 0.0014
- Target probability drop: 0.0014
- Prediction flipped: False

### loo

- Top tokens: compelling (0.4974), of (0.0019), anatomy (0.0014), s (0.0008), ##tti (0.0007), and (0.0003), more (0.0002), ' (0.0001)
- Deleted tokens: compelling
- New probabilities after deletion: negative=0.4991, positive=0.5009
- Confidence drop: 0.4974
- Target probability drop: 0.4974
- Prediction flipped: False

## Validation Example 50

- Sentence: it feels like an after-school special gussied up with some fancy special effects , and watching its rote plot points connect is about as exciting as gazing at an egg timer for 93 minutes . 
- Gold label: negative
- Base prediction: negative
- Base probabilities: negative=0.9949, positive=0.0051
- Case deletion setting: top_k=1.0 (actual_k=1)

### attention

- Top tokens: about (0.2004), is (0.0913), as (0.0868), feels (0.0617), and (0.0449), like (0.0365), ##e (0.0336), as (0.0321)
- Deleted tokens: about
- New probabilities after deletion: negative=0.5112, positive=0.4888
- Confidence drop: 0.4837
- Target probability drop: 0.4837
- Prediction flipped: False

### grad_x_input

- Top tokens: exciting (0.1235), about (0.0757), is (0.0706), , (0.0696), and (0.0517), watching (0.0385), fancy (0.0379), gus (0.0329)
- Deleted tokens: exciting
- New probabilities after deletion: negative=0.9962, positive=0.0038
- Confidence drop: -0.0013
- Target probability drop: -0.0013
- Prediction flipped: False

### loo

- Top tokens: about (0.4837), as (0.0081), is (0.0042), minutes (0.0012), rot (0.0008), timer (0.0007), feels (0.0007), plot (0.0007)
- Deleted tokens: about
- New probabilities after deletion: negative=0.5112, positive=0.4888
- Confidence drop: 0.4837
- Target probability drop: 0.4837
- Prediction flipped: False

## Validation Example 544

- Sentence: although huppert 's intensity and focus has a raw exhilaration about it , the piano teacher is anything but fun . 
- Gold label: negative
- Base prediction: positive
- Base probabilities: negative=0.0015, positive=0.9985
- Case deletion setting: top_k=1.0 (actual_k=1)

### attention

- Top tokens: is (0.2048), fun (0.1501), anything (0.1469), . (0.0633), although (0.0623), but (0.0602), the (0.0584), , (0.0373)
- Deleted tokens: is
- New probabilities after deletion: negative=0.0025, positive=0.9975
- Confidence drop: 0.0010
- Target probability drop: 0.0010
- Prediction flipped: False

### grad_x_input

- Top tokens: but (0.2874), is (0.1331), fun (0.1296), . (0.1121), teacher (0.0592), about (0.0427), focus (0.0398), although (0.0384)
- Deleted tokens: but
- New probabilities after deletion: negative=0.2345, positive=0.7655
- Confidence drop: 0.2330
- Target probability drop: 0.2330
- Prediction flipped: False

### loo

- Top tokens: fun (0.5140), but (0.2330), is (0.0010), anything (0.0006), ##pper (0.0003), has (0.0002), and (0.0001), ##ration (0.0001)
- Deleted tokens: fun
- New probabilities after deletion: negative=0.5155, positive=0.4845
- Confidence drop: 0.4830
- Target probability drop: 0.5140
- Prediction flipped: True

## Validation Example 61

- Sentence: schaeffer has to find some hook on which to hang his persistently useless movies , and it might as well be the resuscitation of the middle-aged character . 
- Gold label: negative
- Base prediction: negative
- Base probabilities: negative=0.9948, positive=0.0052
- Case deletion setting: top_k=1.0 (actual_k=1)

### attention

- Top tokens: useless (0.1306), has (0.1026), be (0.0701), and (0.0579), res (0.0492), ##ation (0.0486), ##ly (0.0452), the (0.0397)
- Deleted tokens: useless
- New probabilities after deletion: negative=0.5136, positive=0.4864
- Confidence drop: 0.4812
- Target probability drop: 0.4812
- Prediction flipped: False

### grad_x_input

- Top tokens: movies (0.2802), persistent (0.1374), of (0.1357), ##ly (0.1340), be (0.1221), useless (0.1082), ##cit (0.0964), res (0.0922)
- Deleted tokens: movies
- New probabilities after deletion: negative=0.9848, positive=0.0152
- Confidence drop: 0.0100
- Target probability drop: 0.0100
- Prediction flipped: False

### loo

- Top tokens: useless (0.4812), movies (0.0100), might (0.0030), has (0.0029), and (0.0025), be (0.0025), ##ly (0.0021), as (0.0014)
- Deleted tokens: useless
- New probabilities after deletion: negative=0.5136, positive=0.4864
- Confidence drop: 0.4812
- Target probability drop: 0.4812
- Prediction flipped: False

## Validation Example 844

- Sentence: given how heavy-handed and portent-heavy it is , this could be the worst thing soderbergh has ever done . 
- Gold label: negative
- Base prediction: negative
- Base probabilities: negative=0.9982, positive=0.0018
- Case deletion setting: top_k=1.0 (actual_k=1)

### attention

- Top tokens: worst (0.2191), thing (0.1202), the (0.1047), done (0.0947), this (0.0797), be (0.0787), has (0.0463), could (0.0270)
- Deleted tokens: worst
- New probabilities after deletion: negative=0.5178, positive=0.4822
- Confidence drop: 0.4804
- Target probability drop: 0.4804
- Prediction flipped: False

### grad_x_input

- Top tokens: worst (0.0285), the (0.0249), thing (0.0218), and (0.0197), this (0.0150), ever (0.0134), so (0.0107), ##nt (0.0105)
- Deleted tokens: worst
- New probabilities after deletion: negative=0.5178, positive=0.4822
- Confidence drop: 0.4804
- Target probability drop: 0.4804
- Prediction flipped: False

### loo

- Top tokens: worst (0.4804), the (0.0003), be (0.0002), this (0.0001), heavy (0.0001), ever (0.0001), done (0.0001), so (0.0001)
- Deleted tokens: worst
- New probabilities after deletion: negative=0.5178, positive=0.4822
- Confidence drop: 0.4804
- Target probability drop: 0.4804
- Prediction flipped: False


## Optional Notes

- To change the experiment suite, edit `SEEDS` and keep `INSPECT_SEED` inside that list.
- Each seed writes to its own directory under `results/checkpoints`, `results/metrics`, and `results/analysis`.
- To reuse existing checkpoints, leave the training cell unexecuted and rerun the evaluation, analysis, or result-loading sections.
- To inspect a different seed's prediction sample or qualitative cases, change `INSPECT_SEED` and rerun the configuration plus result-loading cells.
- To run a smaller smoke test, shorten `SEEDS` or reduce `max_train_examples`, `max_eval_examples`, `sample_size`, and `max_validation_examples` before executing the stage cells.
